In [346]:
import json
import networkx as nx
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity



# Load data

In [347]:
# Load dữ liệu
with open('dataset/alerts_sample.jsonl', 'r', encoding='utf-8') as f:
    alerts = [json.loads(line) for line in f]
with open('dataset/services.json', 'r', encoding='utf-8') as f:
    services_map = json.load(f) # Định nghĩa graph
with open('dataset/incidents_history.json', 'r', encoding='utf-8') as f:
    history = json.load(f)
with open('../d1/results/cluster_summary.json', 'r', encoding='utf-8') as f:
    data = json.load(f) # Đổi tên biến thành 'data' để tránh nhầm lẫn

# Lấy danh sách cluster từ key 'clusters'
cluster_list = data['clusters'] 

print(f"Loaded {len(cluster_list)} clusters for analysis.")

Loaded 2 clusters for analysis.


# Build graph từ services.json

In [348]:
# 2. Khởi tạo Graph
G = nx.DiGraph()

# 3. Thêm tất cả các node (từ list services và stores)
for svc in  services_map['services']:
    G.add_node(svc['name'], type='service', criticality=svc['criticality'])

for store in  services_map['stores']:
    G.add_node(store['name'], type='store', criticality=store['criticality'])

# 4. Thêm các cạnh (edges)
for edge in  services_map['edges']:
    # edge['from'] -> edge['to']
    G.add_edge(edge['from'], edge['to'], type=edge.get('type'))

print(f"Graph built successfully!")
print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")

Graph built successfully!
Nodes: 14, Edges: 17


# Chạy graph + temporal scorer → top-K candidates

In [349]:
def get_alerts_for_cluster(cluster):
    # Danh sách các service thuộc về cluster này
    services_in_cluster = cluster['services']
    
    # Lọc tất cả alert có service nằm trong danh sách trên
    # (Vì file summary đã gom nhóm các service này lại rồi)
    cluster_alerts = [
        a for a in alerts 
        if a['service'] in services_in_cluster
    ]
    
    # Optional: Nếu bạn muốn lọc theo thời gian chặt chẽ hơn, hãy dùng thêm dòng này:
    # cluster_alerts = [a for a in cluster_alerts if cluster['time_range'][0] <= a['ts'] <= cluster['time_range'][1]]
    
    return cluster_alerts

In [350]:
def calculate_rca_score(cluster):
    cluster_services = cluster['services']
    start_time, end_time = cluster['time_range']
 
    subgraph = G.subgraph(cluster_services).copy()
    
    cluster_alerts = [a for a in alerts if a['service'] in cluster_services and start_time <= a['ts'] <= end_time]
    
    if subgraph.number_of_edges() > 0:
        pagerank_scores = nx.pagerank(subgraph, alpha=0.85)
    else:
        pagerank_scores = {svc: 1.0/len(cluster_services) for svc in cluster_services}
    
    # Chuẩn hóa PageRank về [0, 1]
    max_pr = max(pagerank_scores.values()) if pagerank_scores else 1.0
    
    earliest_time = min(a['ts'] for a in cluster_alerts) if cluster_alerts else None
    scores = {}
    
    for svc in cluster_services:
        pr_val = pagerank_scores.get(svc, 0) / max_pr
        out_degree = subgraph.out_degree(svc) if svc in subgraph else 0
        terminal_bonus = 0.05 if out_degree == 0 else 0.0

        svc_alerts = [a for a in cluster_alerts if a['service'] == svc]
        if not svc_alerts:
            t_score = 0.5
        else:
            first_alert = min(a['ts'] for a in svc_alerts)
            t_score = 1.0 if first_alert == earliest_time else 0.5
            
        scores[svc] = (0.6 * pr_val) + (0.4 * t_score) + terminal_bonus
        scores[svc] = min(scores[svc], 1.0)
            
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

print("RCA candidates")
candidates = calculate_rca_score(cluster_list[0])
for svc, score in candidates[:3]:
    print(f"Service: {svc:15} | Score: {score:.4f}")

RCA candidates
Service: payment-svc     | Score: 1.0000
Service: search-svc      | Score: 0.8500
Service: cart-svc        | Score: 0.8411


# Load incidents_history.json → retrieve top-3 similar (keyword similarity)

In [351]:
def get_similar_incidents(cluster, top_k=3):
    # 1. Truy cập vào danh sách incident
    incidents_list = history.get('incidents', [])
    
    scores = []
    cluster_svcs = set(cluster['services'])
    
    for h in incidents_list:
        h_svcs = set(h.get('services_involved', []))
        
        c_score = 0.4 if h.get('root_cause_service') in cluster_svcs else 0.0
        overlap = len(cluster_svcs & h_svcs)
        t_score = min(0.4, 0.2 * overlap)
        s_score = 0.2 if h.get('severity') == cluster.get('severity') else 0.0
        
        total_score = c_score + t_score + s_score
        
        if total_score >= 0.2:
            h_formatted = {
                "incident_id": h.get('id'),
                "root_cause_class": h.get('root_cause_class'),
                "remediation_actions": h.get('remediation', []),
                "summary": h.get('summary')
            }
            scores.append((h_formatted, total_score))
    
    # Sắp xếp theo điểm giảm dần
    return sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

similar_results = get_similar_incidents(cluster_list[0], top_k=1)

if similar_results:
    best_incident, score = similar_results[0]
    
    print("Most similar incident")
    print(f"Incident ID: {best_incident['incident_id']}")
    print(f"Score:       {score:.4f}") # In ra điểm ở đây
    print(f"Summary:     {best_incident['summary']}")
else:
    print("Không tìm thấy incident nào đạt yêu cầu.")

Most similar incident
Incident ID: INC-2025-11-08
Score:       0.8000
Summary:     Payment-svc v3.2 deploy at 09:42 leak DB pool. Pool 50/50 used trong 5 phút. Downstream checkout cascade. Notification queue backed up.


# TF-IDF embedding

In [ ]:
def get_tfidf_similar_incidents(cluster, top_k=3):
    incidents_list = history.get('incidents', [])
    
    corpus = [
        f"{h.get('summary', '')} {' '.join(h.get('services_involved', []))}"
        for h in incidents_list
    ]
    
    query = f"{cluster.get('summary', '')} {' '.join(cluster.get('services', []))}"

    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(corpus + [query])
    
    cosine_sims = cosine_similarity(tfidf_matrix[-1], tfidf_matrix[:-1]).flatten()
    
    results = []
    for idx, score in enumerate(cosine_sims):
        h = incidents_list[idx]
        results.append(({
            "incident_id": h.get('id'),
            "root_cause_class": h.get('root_cause_class'),
            "remediation_actions": h.get('remediation', []),
            "summary": h.get('summary')
        }, float(score)))
        
    return sorted(results, key=lambda x: x[1], reverse=True)[:top_k]

# OUTPUT

In [ ]:
results = []
print(f"{'CLUSTER ID':<12} | {'H-SCORE':<8} | {'T-SCORE':<8} | {'BEST MATCH'}")
for cluster in cluster_list:
    candidates = calculate_rca_score(cluster)
    similar = get_similar_incidents(cluster, top_k=3)
    similar_TFIDF = get_tfidf_similar_incidents(cluster, top_k=3)
    
    cluster_id = cluster.get('cluster_id', 'unknown')
    
    score_h = similar[0][1] if similar else 0.0
    score_t = similar_TFIDF[0][1] if similar_TFIDF else 0.0
    
    if score_h == 0 and score_t == 0:
        # Fallback
        best_match = {
            "incident_id": "NONE",
            "root_cause_class": "unknown",
            "remediation_actions": ["Investigate logs manually", "Check system metrics"]
        }
        used_method = "fallback-no-match"
        final_similar_list = []
        
    elif score_h >= score_t:
        
        best_match = similar[0][0]
        used_method = "graph-heuristic-kNN"
        final_similar_list = [s[0].get('incident_id') for s in similar if s[0].get('incident_id')]
        
    else:
       
        best_match = similar_TFIDF[0][0]
        used_method = "graph-tfidf-semantic"
        final_similar_list = [s[0].get('incident_id') for s in similar_TFIDF if s[0].get('incident_id')]

    top_candidate = candidates[0] if candidates else ("unknown", 0.0)
    
    rca_entry = {
        "cluster_id": cluster_id,
        "graph_top3": candidates[:3] if candidates else [],
        "root_cause": top_candidate[0],
        "class": best_match.get('root_cause_class', 'unknown'),
        "confidence": round(float(top_candidate[1]), 2),
        "actions": best_match.get('remediation_actions', []),
        "reasoning": (
            f"Triggered by {top_candidate[0]} based on graph traversal "
            f"and matched with {best_match.get('incident_id', 'no historical template')}"
        ),
        "similar_incidents": [s[0].get('incident_id') for s in similar if s[0].get('incident_id')],
        "method": "graph-kNN-retrieval"
    }
    results.append(rca_entry)
    print(f"{cluster_id:<12} | {score_h:<8.4f} | {score_t:<8.4f} | {best_match.get('incident_id')}")


with open('results/rca_output.json', 'w', encoding='utf-8') as f:
    json.dump({"clusters_analyzed": len(results), "results": results}, f, indent=2, ensure_ascii=False)

CLUSTER ID   | H-SCORE  | T-SCORE  | BEST MATCH
c-000-000    | 0.8000   | 0.4399   | INC-2025-11-08
c-000-001    | 0.6000   | 0.4613   | INC-2025-08-02
